In [ ]:
# -*- coding: utf-8 -*-
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or
# implied.
# See the License for the specific language governing permissions and
# limitations under the License.
#

# Dimensionality reduction: visual companion

### Author: Hlib Kokin (2026)

This notebook goes with **IAU_03_dimension_reduction**. Each concept has three parts:
1. **Explanation** - the idea, a real-life example, and a question to think about.
2. **Visualization** - the code and the picture.
3. **Conclusion** - what the picture shows.

**Selection vs reduction.** Feature *selection* (notebook 02) **keeps some original columns**. Dimensionality *reduction* **creates new columns**, each a combination of the old ones, so that a few new columns carry most of the information.

> **Example:** a person can be described by 20 body measurements, but most of them move together: tall people have long arms, long legs and big feet. Two new numbers, *"overall size"* and *"build"*, capture most of what the 20 said. Clothing sizes (S/M/L/XL) work the same way.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.manifold import MDS, Isomap, TSNE, trustworthiness
from sklearn.datasets import load_iris, load_digits, make_swiss_roll
from sklearn.preprocessing import StandardScaler
np.set_printoptions(suppress=True, precision=2)

## 1. Matrix factorization and SVD

**The matrix** is simply our data table: rows are objects (users, flowers, documents), columns are features.

**Factorization** means writing something as a product of simpler pieces, like $12 = 3 \times 4$. A table can also be written as a product of smaller tables. Those pieces reveal the **hidden structure** that generated the data.

**SVD (Singular Value Decomposition)** is the most important factorization. It is a proven theorem that **every** table can be written exactly as:

$$R = U \cdot \Sigma \cdot V^T$$

| piece | in our example | meaning |
|---|---|---|
| $U$ | users × tastes | how much each **user** likes each hidden taste |
| $\Sigma$ | strengths | how **important** each taste is, sorted strongest first |
| $V^T$ | tastes × movies | how much each **movie** belongs to each taste |

**Truncated SVD** keeps only the strongest $k$ patterns. That is the compression.

### Example: movie ratings (the idea behind the Netflix Prize)
Six people rate six movies from 0 to 5. Nobody told the computer which movies are action and which are romance.

**Question:** can SVD discover the hidden "tastes" by itself? How many tastes do we need to rebuild the table?

In [ ]:
# ---------- Movie ratings: 6 users x 6 movies (0-5 stars) ----------
movies = ['Die Hard', 'Terminator', 'Mad Max', 'Notebook', 'Titanic', 'La La Land']
users = ['Adam', 'Boris', 'Cyril', 'Dana', 'Eva', 'Filip']
R = np.array([[5, 5, 4, 1, 1, 0],
              [4, 5, 5, 0, 1, 1],
              [5, 4, 5, 1, 0, 1],
              [1, 0, 1, 5, 4, 5],
              [0, 1, 1, 4, 5, 5],
              [3, 3, 3, 3, 3, 3]], dtype=float)

U, S, Vt = np.linalg.svd(R, full_matrices=False)       # R = U @ diag(S) @ Vt
U, Vt = -U, -Vt                                          # flip signs for readability (SVD signs are arbitrary)

fig, axes = plt.subplots(1, 4, figsize=(19, 4.5), gridspec_kw={'width_ratios': [6, 2, 2, 6]})
def heat(ax, M, rows, cols, title, cmap='RdBu_r', vmin=None, vmax=None):
    im = ax.imshow(M, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols, rotation=45, ha='right')
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows)
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            ax.text(j, i, f'{M[i, j]:.1f}' if abs(M[i, j]) < 10 else f'{M[i, j]:.0f}',
                    ha='center', va='center', fontsize=9)
    ax.set_title(title, fontsize=11)

heat(axes[0], R, users, movies, 'R = ratings (users x movies)', cmap='Greens', vmin=0, vmax=5)
heat(axes[1], U[:, :2], users, ['taste 1', 'taste 2'], 'U = user profiles', vmin=-0.7, vmax=0.7)
heat(axes[2], np.diag(S[:2]), ['taste 1', 'taste 2'], ['taste 1', 'taste 2'], 'Σ = strength', cmap='Oranges')
heat(axes[3], Vt[:2], ['taste 1', 'taste 2'], movies, 'Vᵀ = movie profiles', vmin=-0.7, vmax=0.7)
fig.suptitle('R  ≈  U  ×  Σ  ×  Vᵀ    (only the 2 strongest "tastes" shown)', fontsize=13)
plt.tight_layout(); plt.show()

# ---------- How many tastes do we need? ----------
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
energy = S**2 / (S**2).sum()
axes[0].bar([f'#{i+1}' for i in range(len(S))], S, color='tab:orange')
for i, (s, e) in enumerate(zip(S, energy)):
    axes[0].text(i, s + 0.3, f'{e:.1%}', ha='center', fontsize=9)
axes[0].set_title('Singular values (strength of each pattern)\n% = share of total information', fontsize=11)
for ax, k in zip(axes[1:], [1, 2, 6]):
    Rk = (U[:, :k] * S[:k]) @ Vt[:k]
    heat(ax, Rk, users, movies, f'rebuilt from top {k} pattern(s)\nmean error = {np.abs(R - Rk).mean():.2f} stars',
         cmap='Greens', vmin=0, vmax=5)
plt.tight_layout(); plt.show()

### Conclusion
- **SVD found the two tastes by itself.**
  - In $V^T$, taste 2 is **negative for action movies** (Die Hard, Terminator, Mad Max) and **positive for romance** (Notebook, Titanic, La La Land). That is the action-vs-romance axis.
  - In $U$, Adam, Boris and Cyril (action fans) are negative on taste 2; Dana and Eva (romance fans) are positive. Filip, who rates everything 3, is near zero.
  - Taste 1 is roughly "likes movies in general".
  - The **sign** of a taste is arbitrary; only the grouping matters.
- **Singular values:** the first two patterns hold **69.6% + 28.9% = 98.5%** of the information. The remaining four are almost nothing.
- **Rebuilding the table:**

  | patterns kept | mean error |
  |---|---|
  | 1 | 1.61 stars |
  | **2** | **0.35 stars** |
  | 6 | 0.00 stars (exact) |

  With 2 patterns, 6 columns compress into 2 almost without loss.

**Where this is used:**
- **Recommender systems:** missing ratings are predicted from the user and movie profiles.
- **LSA (Latent Semantic Analysis):** the same thing on a documents × words table, where the hidden patterns are *topics*.
- **Image compression.**

## 2. PCA (Principal Component Analysis)

**Idea: the best shadow.** Hold a pencil under a lamp. Lit from its end, the shadow is a dot and you lose everything. Lit from the side, the shadow shows its full length. PCA finds the angle where the data's "shadow" (projection) is **most spread out**:
- **PC1** is the direction of **maximum variance**.
- **PC2** is the next best direction, **at a right angle** to PC1, and so on.

Each principal component is a **weighted combination of the original features**, for example:

$$PC1 = 0.36\cdot\text{sepal length} - 0.08\cdot\text{sepal width} + 0.86\cdot\text{petal length} + 0.36\cdot\text{petal width}$$

**Explained variance ratio** = how much of the total variation each component keeps.

### Examples
1. **Height and arm span** of 200 people. The two are almost the same number, so one direction should describe nearly everything.
2. **Iris:** what is inside PC1 and PC2, and how many components we need.

**Question:** how much information do we lose if we describe each person by one number instead of two?

In [ ]:
# ---------- PCA on 2 body measurements: height and arm span ----------
rng = np.random.default_rng(1)
height = rng.normal(175, 9, 200)
arm_span = height + rng.normal(0, 3, 200)                  # arm span ~ height
B = np.column_stack([height, arm_span])
pca = PCA(2).fit(B)
Z = pca.transform(B)
proj = pca.inverse_transform(np.column_stack([Z[:, 0], np.zeros(len(Z))]))   # shadow on PC1

fig, axes = plt.subplots(1, 3, figsize=(19, 5.5))
ax = axes[0]
for p, q in zip(B[:40], proj[:40]):
    ax.plot([p[0], q[0]], [p[1], q[1]], color='lightgrey', lw=1)        # projection lines
ax.scatter(*B.T, s=12, color='tab:blue', alpha=0.6, label='people')
ax.scatter(*proj[:40].T, s=12, color='tab:red', label='"shadow" on PC1')
m = pca.mean_
for comp, var, name, col in zip(pca.components_, pca.explained_variance_, ['PC1', 'PC2'], ['tab:red', 'tab:green']):
    v = comp * 2.5 * np.sqrt(var)
    ax.annotate('', xy=m + v, xytext=m, arrowprops=dict(color=col, lw=3, arrowstyle='->'))
    ax.text(*(m + v * 1.1), name, color=col, fontsize=12, fontweight='bold')
ax.set_aspect('equal'); ax.set_xlabel('height (cm)'); ax.set_ylabel('arm span (cm)'); ax.legend(loc='upper left')
r = pca.explained_variance_ratio_
ax.set_title(f'PC1 = "overall size" ({r[0]:.0%} of variance)\nPC2 = "arms longer/shorter than height" ({r[1]:.0%})', loc='left')

# ---------- Iris: what is inside each component? ----------
X_iris, y_iris = load_iris(return_X_y=True)
names = ['sepal length', 'sepal width', 'petal length', 'petal width']
p4 = PCA(4).fit(X_iris)
ax = axes[1]
xs = np.arange(4)
ax.bar(xs - 0.2, p4.components_[0], 0.4, label='PC1')
ax.bar(xs + 0.2, p4.components_[1], 0.4, label='PC2')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(xs); ax.set_xticklabels(names, rotation=20); ax.legend()
ax.set_title('Iris: weight of each original feature\ninside PC1 and PC2', loc='left')

ax = axes[2]
cum = np.cumsum(p4.explained_variance_ratio_)
ax.bar(range(1, 5), p4.explained_variance_ratio_, color='tab:purple', label='each component')
ax.plot(range(1, 5), cum, 'o-', color='black', label='cumulative')
for i, (v, c) in enumerate(zip(p4.explained_variance_ratio_, cum)):
    ax.text(i + 1, v + 0.02, f'{v:.1%}', ha='center')
    if i > 0: ax.text(i + 1, c + 0.03, f'{c:.1%}', ha='center')
ax.set_xticks(range(1, 5)); ax.set_xlabel('component'); ax.set_ylim(0, 1.1); ax.legend(loc='center right')
ax.set_title('Iris: how much each component explains\n("scree plot")', loc='left')
plt.tight_layout(); plt.show()

### Conclusion
- **Height and arm span:**
  - **PC1 = "overall size"** and explains **98%** of the variance.
  - PC2 = "arms longer or shorter than height" explains only 2%.
  - The red points are the people's "shadows" on PC1. Describing each person by **one number instead of two loses only 2%** of the information.
- **Iris weights (middle plot):**
  - **PC1** is dominated by **petal length** (0.86), plus some sepal length and petal width. It is basically "flower size".
  - **PC2** is mostly **sepal length and sepal width**, so it is "sepal size".
- **Scree plot (right):** PC1 explains **92.5%**, and PC1 + PC2 together **97.8%**. 4 columns can become 2 with almost no loss.
- **How to choose the number of components:** keep enough to reach, for example, 95% cumulative variance, or cut where the bars suddenly become small (the "elbow"). This is a common rule of thumb, not a law.

**Other real uses:** body sizing systems, a "market factor" in stock returns, image compression ("eigenfaces"), and removing noise before another model.

## 3. Truncated SVD vs PCA: why the notebook gives different numbers

In the original notebook, iris gives:
- `TruncatedSVD`: explained variance **[0.53, 0.45]**
- `PCA`: explained variance **[0.92, 0.05]**

Why so different on the same data?

**PCA = centre the data (subtract each column's mean), then SVD.** Truncated SVD **skips the centring**.

**Question:** iris values are all positive (1-8 cm), far from the point (0, 0). Without centring, what will the first direction of SVD point at?

In [ ]:
# ---------- 2D toy data far from the origin (like iris: all values positive) ----------
rng = np.random.default_rng(0)
t = rng.normal(size=300)
X2 = np.column_stack([10 + 2.0 * t + rng.normal(0, 0.4, 300),      # e.g. petal length
                      6 - 1.0 * t + rng.normal(0, 0.4, 300)])      # e.g. something correlated

svd_dir = TruncatedSVD(1).fit(X2).components_[0]
pca = PCA(2).fit(X2); pca_dir = pca.components_[0]
mean = X2.mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
ax = axes[0]
ax.scatter(*X2.T, s=8, alpha=0.5, color='grey')
ax.scatter([0], [0], color='black', s=60); ax.text(0.3, -0.8, 'origin (0,0)')
ax.scatter(*mean, color='red', s=60, zorder=3); ax.text(mean[0] + 0.3, mean[1] + 0.5, 'mean of data', color='red')
L = 8
ax.annotate('', xy=svd_dir * 14, xytext=(0, 0), arrowprops=dict(color='tab:blue', lw=3, arrowstyle='->'))
ax.text(*(svd_dir * 14 + [0.3, 0.6]), 'Truncated SVD\n(no centring): points\nat the data from (0,0)', color='tab:blue')
s = np.sign(pca_dir[0]) if pca_dir[0] != 0 else 1
ax.annotate('', xy=mean + s * pca_dir * L, xytext=mean - s * pca_dir * L,
            arrowprops=dict(color='tab:green', lw=3, arrowstyle='<->'))
ax.text(*(mean + s * pca_dir * L + [-3, -2.5]), 'PCA (centred first):\nalong the direction\nwhere the data VARIES', color='tab:green')
ax.set_aspect('equal'); ax.set_xlim(-2, 20); ax.set_ylim(-4, 12); ax.grid(alpha=0.3)
ax.set_title('Same data, first direction found by each method', loc='left')

# iris: explained variance of each method
X_iris, y_iris = load_iris(return_X_y=True)
res = {'Truncated SVD\n(raw)': TruncatedSVD(2).fit(X_iris).explained_variance_ratio_,
       'PCA\n(raw)': PCA(2).fit(X_iris).explained_variance_ratio_,
       'PCA\n(standardized)': PCA(2).fit(StandardScaler().fit_transform(X_iris)).explained_variance_ratio_}
ax = axes[1]
xs = np.arange(len(res))
ax.bar(xs - 0.2, [v[0] for v in res.values()], 0.4, label='component 1')
ax.bar(xs + 0.2, [v[1] for v in res.values()], 0.4, label='component 2')
for i, v in enumerate(res.values()):
    ax.text(i - 0.2, v[0] + 0.01, f'{v[0]:.2f}', ha='center'); ax.text(i + 0.2, v[1] + 0.01, f'{v[1]:.2f}', ha='center')
ax.set_xticks(xs); ax.set_xticklabels(res.keys()); ax.set_ylim(0, 1.05); ax.legend()
ax.set_ylabel('explained variance ratio'); ax.set_title('Iris: what the notebook printed (+ standardized PCA)', loc='left')
plt.tight_layout(); plt.show()

### Conclusion
- **Without centring** (blue arrow), the first direction points from the origin **towards the data cloud**. It describes "how far the data is from zero", not how the points *differ* from each other.
- **With centring** (green arrow, PCA), the first direction runs **along the cloud**, where the points actually vary. That is what we want for compression.
- So on iris, PCA's first component explains 92% of the variation, while truncated SVD's numbers mean something else.

**Why truncated SVD exists anyway:** for huge **sparse** tables, such as millions of documents × words that are mostly zeros, centring would turn every zero into a non-zero number and make the table too big for memory. For text, truncated SVD (LSA) is the right tool. For ordinary dense tables, use PCA.

**Scaling:** PCA follows variance, so units matter. Standardized iris gives **[0.73, 0.23]** instead of [0.92, 0.05]. Standardize first unless all features are in the same unit.

## 4. LDA (Linear Discriminant Analysis) vs PCA

PCA is **unsupervised**: it never looks at the labels, it just follows the biggest spread.

LDA is **supervised**: it uses the labels and looks for the direction that **best separates the classes**:

$$\text{maximize}\quad \frac{\text{between-class variance}}{\text{within-class variance}}$$

This is exactly the **ANOVA F idea** from notebook 02. The difference: ANOVA *scores* existing features, while LDA *builds* the best combination of features.

### Example: apples vs oranges
Fruits vary a lot in **size**, but size does not tell apples from oranges. A smaller difference, **colour**, does. The toy data below is built exactly like this: a big spread that is the same for both classes, and a small gap that separates them.

**Question:** which direction will PCA choose, and which will LDA choose? After squashing each fruit to one number, will the classes still be separable?

In [ ]:
# ---------- Toy case where the biggest spread is NOT the useful direction ----------
rng = np.random.default_rng(2)
n = 200
long_axis = rng.normal(0, 6, 2 * n)                       # big spread, same for both classes
short_axis = np.r_[rng.normal(-1.5, 0.6, n), rng.normal(1.5, 0.6, n)]   # small spread, separates classes
angle = np.deg2rad(30)
Rot = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
T = np.column_stack([long_axis, short_axis]) @ Rot.T
yt = np.r_[np.zeros(n), np.ones(n)].astype(int)

pca1 = PCA(1).fit(T); lda1 = LinearDiscriminantAnalysis(n_components=1).fit(T, yt)
zp, zl = pca1.transform(T).ravel(), lda1.transform(T).ravel()
colors = np.array(['tab:red', 'tab:green'])

fig, axes = plt.subplots(2, 3, figsize=(19, 10))
ax = axes[0, 0]
ax.scatter(*T.T, c=colors[yt], s=8, alpha=0.6)
c = T.mean(0)
dp = pca1.components_[0]; dl = lda1.scalings_[:, 0] / np.linalg.norm(lda1.scalings_[:, 0])
ax.annotate('', xy=c + 14 * dp, xytext=c - 14 * dp, arrowprops=dict(color='tab:blue', lw=3, arrowstyle='<->'))
ax.text(*(c + 14 * dp), ' PCA', color='tab:blue', fontsize=12, fontweight='bold')
ax.annotate('', xy=c + 5 * dl, xytext=c - 5 * dl, arrowprops=dict(color='black', lw=3, arrowstyle='<->'))
ax.text(*(c + 5 * dl), ' LDA', color='black', fontsize=12, fontweight='bold')
ax.set_aspect('equal'); ax.set_title('Two classes (red = apples, green = oranges)\nPCA follows the biggest spread, LDA the class gap', loc='left')
for ax, z, name in [(axes[0, 1], zp, 'PCA (ignores labels)'), (axes[0, 2], zl, 'LDA (uses labels)')]:
    for k in (0, 1):
        ax.hist(z[yt == k], bins=30, alpha=0.6, color=colors[k])
    ax.set_title(f'Data squashed onto the {name} direction\n-> 1 number per fruit', loc='left')
    ax.set_yticks([])

# ---------- Iris: PCA vs LDA in 2D ----------
X_iris, y_iris = load_iris(return_X_y=True)
cols3 = ['navy', 'turquoise', 'darkorange']; names3 = load_iris().target_names
for ax, Z, title in [(axes[1, 0], PCA(2).fit_transform(X_iris), 'Iris - PCA (labels NOT used)'),
                     (axes[1, 1], LinearDiscriminantAnalysis(n_components=2).fit_transform(X_iris, y_iris),
                      'Iris - LDA (labels used)')]:
    for k in range(3):
        ax.scatter(*Z[y_iris == k].T, s=15, color=cols3[k], label=names3[k], alpha=0.8)
    ax.legend(); ax.set_title(title, loc='left')
axes[1, 2].axis('off')
axes[1, 2].text(0, 0.5,
    "PCA: direction of MAXIMUM VARIANCE\n"
    "     (does not know the classes)\n\n"
    "LDA: direction of MAXIMUM\n"
    "     between-class / within-class\n"
    "     (= the ANOVA F idea!)\n\n"
    "LDA gives at most (classes - 1)\n"
    "components: 2 classes -> 1, iris -> 2",
    fontsize=12, family='monospace', va='center')
plt.tight_layout(); plt.show()

### Conclusion
- **The toy fruit example:**
  - **PCA** chose the direction of the **biggest spread** (size). Projected onto it, the red and green fruits **overlap completely**, so the one number is useless for classification.
  - **LDA** chose the direction of the **class gap** (colour). Projected onto it, the two classes are **clearly separated**.
  - **The biggest variance is not always the useful information.**
- **Iris:** both methods separate setosa. LDA separates versicolor and virginica a bit better, because it was told which flower is which.
- **Facts to remember:**
  - LDA gives **at most (number of classes − 1)** components: 2 classes give 1, iris gives 2. This is mathematically fixed.
  - LDA needs labels, so it must be **fitted on the training data only**.
  - Don't confuse it with *Latent Dirichlet Allocation*, a topic model with the same abbreviation.

## 5. MDS (Multidimensional Scaling)

**Goal:** place points in 2D so that the **distances between them match the original distances** as closely as possible.

MDS does not need features at all. It only needs a **table of distances** (or dissimilarities) between objects.

### Example: rebuild a map of Central Europe from a distance table
Below is a table of distances between 11 cities. **MDS gets only this table**, with no coordinates.

**Question:** can MDS draw the map? What can it **not** know from distances alone?

**Other real uses:**
- **Marketing:** people rate how similar brands are, and MDS draws a "perceptual map".
- **Biology:** similarity between species or proteins.

In [ ]:
# ---------- Only a DISTANCE TABLE between cities -> can MDS draw the map? ----------
from scipy.spatial import procrustes
cities = {'Bratislava': (48.15, 17.11), 'Vienna': (48.21, 16.37), 'Prague': (50.08, 14.44),
          'Budapest': (47.50, 19.04), 'Warsaw': (52.23, 21.01), 'Berlin': (52.52, 13.40),
          'Munich': (48.14, 11.58), 'Krakow': (50.06, 19.94), 'Ljubljana': (46.06, 14.51),
          'Zagreb': (45.81, 15.98), 'Kosice': (48.72, 21.26)}
names = list(cities); lat, lon = np.array(list(cities.values())).T

def km(lat1, lon1, lat2, lon2):                 # great-circle distance
    p1, p2, dl = np.radians(lat1), np.radians(lat2), np.radians(lon2 - lon1)
    return 6371 * np.arccos(np.clip(np.sin(p1) * np.sin(p2) + np.cos(p1) * np.cos(p2) * np.cos(dl), -1, 1))
D = np.array([[km(lat[i], lon[i], lat[j], lon[j]) for j in range(len(names))] for i in range(len(names))])
display(pd.DataFrame(D.round(0).astype(int), index=names, columns=names))   # <- this is ALL MDS gets

import warnings
with warnings.catch_warnings():               # silence sklearn's "parameter renamed" notices
    warnings.simplefilter('ignore', FutureWarning)
    mds = MDS(n_components=2, dissimilarity='precomputed', random_state=0, n_init=4, init='random')
    P = mds.fit_transform(D)

true_xy = np.column_stack([lon * np.cos(np.radians(lat.mean())), lat])   # simple map projection
_, P_aligned, _ = procrustes(true_xy, P)          # rotate/flip/scale MDS result to compare
true_n, _, _ = procrustes(true_xy, true_xy)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, XY, title in [(axes[0], true_n, 'REAL map (from GPS coordinates)'),
                      (axes[1], P_aligned, 'MDS map (from the distance table ONLY)')]:
    ax.scatter(*XY.T, color='tab:red', s=40)
    for nme, (a, b) in zip(names, XY):
        ax.text(a, b + 0.012, nme, ha='center', fontsize=10)
    ax.set_title(title, loc='left'); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect('equal')
plt.tight_layout(); plt.show()
D_mds = np.linalg.norm(P[:, None] - P[None], axis=2)
print(f'average error of MDS distances vs real distances: {np.abs(D_mds - D)[np.triu_indices(len(names), 1)].mean():.1f} km')

### Conclusion
- From the distance table alone, MDS reconstructed **the map of Central Europe**. The average error of the MDS distances is **under 1 km**.
- **What MDS cannot know:** north/south, east/west, and mirror images. Distances do not change if you rotate or flip a map. That is why the result was rotated and flipped to match the real map before plotting (the `procrustes` step).
- MDS compares **all pairs** of points, so it is **slow** for large datasets. That is why the original notebook uses only 100 digit images.
- **Established fact:** classical MDS on ordinary straight-line distances gives the same result as PCA. MDS becomes really different when the distances are something else: road distances, survey similarities, or distances along a surface (next section).

## 6. Manifold learning and Isomap

**Problem:** PCA, LDA and classical MDS are **linear**. They cast a flat shadow, which fails when data lies on a **curved surface**.

**Manifold:** a shape that looks flat when you zoom in, but is curved overall.
- The **Earth's surface** sits in 3D, but every place is described by **2 numbers** (latitude, longitude).
- A **rolled-up carpet ("Swiss roll")** is a 2D sheet rolled up in 3D.
- **Photos of a turning face** have thousands of pixels, but really only **1 number** (the angle) changes.

**Manifold learning** tries to **unroll** such shapes.

**Isomap (Isometric Mapping)** measures distance **along the surface**, like an ant walking on the carpet:
1. Connect each point to its nearest neighbours. Short hops are trustworthy because locally the sheet is flat.
2. Compute the shortest path between all pairs **through this network**. This is the *geodesic distance*.
3. Run MDS on those distances.

**Embedding** = the new low-dimensional coordinates you get for each point.

**Established vs assumption:** that *some* data lies on curved low-dimensional surfaces is demonstrable (the Swiss roll below). That *most* real high-dimensional data does (the "manifold hypothesis") is a widely used, well-supported **working assumption**, not a proven law.

**Question:** which methods will manage to unroll the carpet? In the 2D result, the colours should go in order from one side to the other.

In [ ]:
# ---------- Swiss roll: a 2D sheet rolled up in 3D ----------
Xs, color = make_swiss_roll(n_samples=1500, noise=0.05, random_state=0)   # color = position along the sheet
fig = plt.figure(figsize=(19, 10))

ax = fig.add_subplot(2, 3, 1, projection='3d')
ax.scatter(*Xs.T, c=color, cmap='Spectral', s=5)
ax.view_init(10, -70); ax.set_title('Swiss roll in 3D\n(colour = position along the unrolled sheet)', loc='left')

# straight-line vs along-the-sheet distance for two points
a = np.argmin(np.abs(color - 6) + np.abs(Xs[:, 1] - 10)); b = np.argmin(np.abs(color - 12.5) + np.abs(Xs[:, 1] - 10))
ax = fig.add_subplot(2, 3, 2)
ax.scatter(Xs[:, 0], Xs[:, 2], c=color, cmap='Spectral', s=5)
ax.plot(*Xs[[a, b]][:, [0, 2]].T, 'k--', lw=2, label='straight line (what MDS/PCA "see")')
tt = np.linspace(color[a], color[b], 200)                     # the sheet: x = t*cos(t), z = t*sin(t)
ax.plot(tt * np.cos(tt), tt * np.sin(tt), color='black', lw=3, label='along the sheet (what Isomap measures)')
ax.legend(loc='lower left', fontsize=8)
ax.scatter(*Xs[[a, b]][:, [0, 2]].T, color='black', s=80, zorder=3)
straight = np.linalg.norm(Xs[a] - Xs[b])
along = np.sum(np.hypot(np.diff(tt * np.cos(tt)), np.diff(tt * np.sin(tt))))
ax.set_title(f'View from the side: two points\nstraight line = {straight:.1f},  along the sheet = {along:.1f}\n'
             f'(an ant on the carpet must walk around the roll)', loc='left')
ax.set_aspect('equal')

methods = [('PCA (linear "shadow")', PCA(2)),
           ('Isomap, 10 neighbours', Isomap(n_neighbors=10, n_components=2)),
           ('MDS (straight-line distances)', MDS(2, random_state=0, n_init=1, init='random')),
           ('Isomap, 60 neighbours (too many!)', Isomap(n_neighbors=60, n_components=2))]
idx = np.random.default_rng(0).choice(len(Xs), 600, replace=False)      # MDS is slow: use a subset for all
for k, (title, m) in enumerate(methods):
    E = m.fit_transform(Xs[idx])
    ax = fig.add_subplot(2, 3, 3 + k)
    ax.scatter(*E.T, c=color[idx], cmap='Spectral', s=6)
    ax.set_title(title + '\n' + ('colours in order = sheet unrolled' if k == 1 else 'colours mixed/overlapping = not unrolled'),
                 loc='left')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### Conclusion
- **Two points on neighbouring layers** are only **7.7** apart in a straight line, but **62.9** apart along the sheet. Methods that use straight-line distances think they are neighbours, which they are not.
- **PCA and MDS** (straight-line distances) **could not unroll** the carpet: the colours stay mixed and wrapped.
- **Isomap with 10 neighbours unrolled it**: the colours go in order from red to blue, a flat rectangle, just like the real sheet.
- **Isomap with 60 neighbours failed.** With too many neighbours, the network connects points across neighbouring layers ("short circuits"), and distances along the sheet become wrong again. **The number of neighbours is a critical setting.**

## 7. Real data: handwritten digits - and t-SNE

The original notebook switches to the **digits** dataset: images of 8×8 pixels, so **64 features** per image, and 10 classes (0-9).

**t-SNE (t-distributed Stochastic Neighbor Embedding)** has one goal: **keep neighbours together**. It cares about *local* structure ("who is close to whom"), not about large distances.

**Analogy: a wedding seating plan.** The planner only makes sure everyone sits near their closest friends. Whether the family table is 3 m or 10 m from the colleagues' table does not matter.

**How it works:**
1. In the original 64D space, turn distances into a "probability of being neighbours", using a bell curve. **Perplexity** is roughly how many neighbours each point considers.
2. In 2D, do the same with a **Student's t-distribution**. Its heavier tails give the map more room, so clusters do not get crushed into the middle.
3. Move the 2D points step by step until both sets of neighbour probabilities match. The mismatch is measured by *KL divergence*.

**How we compare the methods: trustworthiness.** Are a point's neighbours on the 2D map really its neighbours in the original 64D? **1.0 = perfectly**, lower = the map shows false neighbours.

**Question:** the labels are **never** given to these methods (they are only used for colouring). Which method will separate the 10 digits best?

In [ ]:
# ---------- The data: handwritten digits, 8x8 pixels = 64 features ----------
Xd, yd = load_digits(return_X_y=True)
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for d in range(10):
    axes[d].imshow(Xd[yd == d][0].reshape(8, 8), cmap='gray_r'); axes[d].set_title(str(d)); axes[d].axis('off')
plt.suptitle('Each image = 64 numbers (pixel brightness 0-16). Can we draw 1797 images on a 2D map?', y=1.1)
plt.show()

# ---------- All methods on the same 500 images ----------
Xs5, ys5 = Xd[:500], yd[:500]
methods = [('PCA', PCA(2)),
           ('MDS', MDS(2, random_state=0, n_init=1, init='random')),
           ('Isomap', Isomap(n_neighbors=10, n_components=2)),
           ('t-SNE', TSNE(2, random_state=0))]
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, (name, m) in zip(axes, methods):
    E = m.fit_transform(Xs5)
    sc = ax.scatter(*E.T, c=ys5, cmap='tab10', s=10)
    for d in range(10):                                   # write the digit at its cluster centre
        ax.text(*np.median(E[ys5 == d], axis=0), str(d), fontsize=16, fontweight='bold',
                ha='center', va='center', bbox=dict(facecolor='white', alpha=0.7, lw=0))
    ax.set_title(f'{name}\ntrustworthiness = {trustworthiness(Xs5, E, n_neighbors=5):.3f}', loc='left')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### Conclusion

| method | trustworthiness | what the picture shows |
|---|---|---|
| PCA | 0.854 | digits overlap heavily: a flat shadow loses a lot |
| MDS | 0.897 | slightly better, still overlapping |
| Isomap | 0.914 | clearer groups |
| **t-SNE** | **0.996** | **10 clearly separated islands, one per digit** |

- None of the methods saw the labels, yet t-SNE grouped the images by digit almost perfectly. Images of the same digit really are close to each other in the 64D space, and t-SNE is best at showing that.
- That is why t-SNE (and its newer cousin **UMAP**) is the standard tool for **looking at** high-dimensional data: images, word embeddings, single-cell biology.

## 8. How (not) to read a t-SNE picture

t-SNE pictures are beautiful but **easy to misread**. Below: the same data and the same method, with
- different **perplexity** (how many neighbours each point considers: 5, 30, 100), and
- a different **random start** (`init='random'`, `seed=1`).

**Question:** what changes between the pictures, and what stays the same?

In [ ]:
# ---------- t-SNE: the picture depends on perplexity and on the random seed ----------
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
settings = [(5, 0, 'pca'), (30, 0, 'pca'), (100, 0, 'pca'), (30, 1, 'random')]
for ax, (perp, seed, init) in zip(axes, settings):
    E = TSNE(2, perplexity=perp, random_state=seed, init=init).fit_transform(Xs5)
    ax.scatter(*E.T, c=ys5, cmap='tab10', s=10)
    for d in range(10):
        ax.text(*np.median(E[ys5 == d], axis=0), str(d), fontsize=14, fontweight='bold',
                ha='center', va='center', bbox=dict(facecolor='white', alpha=0.7, lw=0))
    ax.set_title(f'perplexity = {perp}, seed = {seed}, init = {init}', loc='left')
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('Same data, same method - different pictures. Cluster positions, sizes and gaps change; '
             'WHICH points are neighbours stays.', y=1.03, fontsize=12)
plt.tight_layout(); plt.show()

### Conclusion

**What stays the same:** **which points are neighbours.** The same digits form the same groups in every picture.

**What changes:** the **positions** of clusters, their **sizes**, and the **gaps between them**.
- With perplexity 5, the clusters break into small fragments.
- With perplexity 100, they become wider and closer.
- A different random start places the clusters in completely different spots.

**Rules for reading t-SNE (well documented; the most common mistakes):**
1. **Distances between clusters mean nothing.** "0 is far from 7" on one picture is not a fact about the data.
2. **Cluster sizes mean nothing.** A big blob is not necessarily a more varied group.
3. **Try several perplexity values** before drawing conclusions.
4. **There is no `transform()` for new data**, and the result is random. Use t-SNE for **visualization**, not as a preprocessing step for a model.

## Summary

| method | linear? | uses labels? | preserves | typical use |
|---|---|---|---|---|
| **Truncated SVD** | yes | no | main patterns (no centring) | huge sparse data: text (LSA), recommendations |
| **PCA** | yes | no | maximum variance | general compression, preprocessing, noise removal |
| **LDA** | yes | **yes** | class separation | compression before classification (max classes − 1 dims) |
| **MDS** | no* | no | all pairwise distances | maps from distance / similarity tables |
| **Isomap** | no | no | distances **along** the surface | unrolling curved data |
| **t-SNE** | no | no | **local** neighbourhoods | visualization only |

\*Classical MDS on straight-line distances = PCA.

**Always:** fit the reduction on **training data only**. PCA learns means and directions, and LDA also learns from the labels.